# Signal Detection Theory Analysis — Experiment 1

Computes **d'** (sensitivity), **criterion c** (response bias), and **AUC** for each model × memory condition using:
- `sklearn.metrics.confusion_matrix` — extract TP/FP/TN/FN
- `sklearn.metrics.roc_auc_score`, `roc_curve` — non-parametric ROC analysis
- `scipy.stats.norm.ppf` — Z-score transformation for d' and c

Signal = *imagined* (internal) | Noise = *perceived* (external)

**Figures exported:** FigS7 (d', c, AUC bars) · FigS8 (ROC curves)

**Referenced in:** Supplemental Materials, Section S3

In [ ]:
# ── Imports & project config ───────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import norm

# sklearn — SDT computation & ROC
from sklearn.metrics import (confusion_matrix, roc_auc_score,
                             roc_curve, ConfusionMatrixDisplay)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D

import rmllm
PROJ = Path(rmllm.config.PROJ_ROOT)
DATA = PROJ / 'data' / 'processed'
SUP  = PROJ / 'reports' / 'figures' / 'supplemental'
SUP.mkdir(parents=True, exist_ok=True)

DPI = 700

# ── Shared aesthetics (matches manuscript figures) ─────────────────────────
plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 11,
    'axes.titlesize': 12, 'axes.titleweight': 'bold',
    'axes.labelsize': 11, 'axes.labelweight': 'bold',
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'axes.linewidth': 1.0, 'axes.facecolor': 'white',
    'figure.facecolor': 'white', 'axes.grid': False,
    'xtick.bottom': True, 'ytick.left': True,
    'xtick.direction': 'out', 'ytick.direction': 'out',
    'xtick.major.size': 4, 'ytick.major.size': 4,
    'legend.fontsize': 9, 'legend.framealpha': 0.9,
    'legend.edgecolor': '#cccccc',
})

MODEL_ORDER  = ['Gemma3:12b','Gemma3:12b-QAT','Gemma3:27b',
                'Gemma3:27b-QAT','Llama3.3:70b','Llama4:16x17b']
MODEL_LABELS = ['G3:12b','G3:12b\nQAT','G3:27b',
                'G3:27b\nQAT','L3.3:70b','L4:16x17b']
MODEL_COLORS = plt.cm.tab10.colors[:6]

def _panel_tag(ax, letter, title=''):
    ax.text(-0.13, 1.07, letter, transform=ax.transAxes,
            fontsize=14, fontweight='bold', va='top', ha='left')
    if title:
        ax.set_title(title, pad=6, fontsize=12, fontweight='bold')

print("Setup complete.")


In [ ]:
# ── SDT helper functions (sklearn + scipy) ────────────────────────────────
def compute_sdt(y_true, y_pred, y_score=None, accuracy=None, confidence=None):
    """
    Compute Type 1 and Type 2 SDT measures for one model/condition.

    Type 1 (source discrimination):
      y_true  : 1 = signal (imagined), 0 = noise (perceived)
      y_pred  : 1 = responded 'internal', 0 = responded 'external'
      y_score : signed confidence score for Type 1 ROC/AUC
                (internal judgment = +confidence, external = -confidence)

    Type 2 (metacognitive sensitivity):
      accuracy   : 1 = correct judgment, 0 = incorrect judgment
      confidence : raw confidence rating (higher = more certain)
      Type 2 AUC (roc_auc_score(accuracy, confidence)) measures how well
      confidence tracks correctness, i.e. metacognitive sensitivity.
      Undefined (NaN) when accuracy is constant (all correct or all wrong).

    Returns dict with keys:
      HR, FA, dprime, c, auc1 (Type 1), auc2 (Type 2), n_signal, n_noise
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    # sklearn confusion_matrix: rows=true, cols=pred → [TN FP / FN TP]
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    TN, FP, FN, TP = cm.ravel()

    n_signal = TP + FN   # total imagined trials
    n_noise  = TN + FP   # total perceived trials

    # Log-linear correction (Hautus 1995) — avoids ±inf with HR=1 or FA=0
    HR = (TP + 0.5) / (n_signal + 1)
    FA = (FP + 0.5) / (n_noise  + 1)

    dprime = float(norm.ppf(HR) - norm.ppf(FA))
    c      = float(-0.5 * (norm.ppf(HR) + norm.ppf(FA)))

    # ── Type 1 AUC (source discrimination via signed confidence) ──────────
    auc1 = None
    if y_score is not None and len(np.unique(y_score)) > 1:
        try:
            auc1 = roc_auc_score(y_true, y_score)
        except Exception:
            auc1 = np.nan

    # ── Type 2 AUC (metacognitive: does confidence track correctness?) ─────
    # roc_auc_score(correct/incorrect, confidence)
    # NaN when all trials are correct or all wrong (no variance in accuracy)
    auc2 = None
    if accuracy is not None and confidence is not None:
        acc_arr  = np.asarray(accuracy)
        conf_arr = np.asarray(confidence, dtype=float)
        if len(np.unique(acc_arr)) > 1 and len(np.unique(conf_arr)) > 1:
            try:
                auc2 = roc_auc_score(acc_arr, conf_arr)
            except Exception:
                auc2 = np.nan
        else:
            auc2 = np.nan   # degenerate: all correct or all wrong

    return dict(HR=round(HR,4), FA=round(FA,4),
                dprime=round(dprime,4), c=round(c,4),
                auc1=round(auc1,4) if auc1 is not None else None,
                auc2=round(auc2,4) if (auc2 is not None and not np.isnan(auc2)) else np.nan,
                n_signal=int(n_signal), n_noise=int(n_noise))


def signed_confidence(judgment, confidence):
    """
    Signed discriminant score for Type 1 ROC.
    Internal judgment → +confidence; External → -confidence.
    """
    return np.where(judgment == 'internal',
                    confidence.astype(float),
                   -confidence.astype(float))


print("SDT functions defined (Type 1 + Type 2).")


In [ ]:
# ── Load & preprocess Exp 1 data ──────────────────────────────────────────
df = pd.read_csv(DATA / 'exp1_trial_data.csv')

# Binary labels: imagined = signal (1), perceived = noise (0)
df['y_true'] = (df['source'] == 'imagined').astype(int)
df['y_pred'] = (df['Judgment'] == 'internal').astype(int)

# Signed confidence for ROC (higher = more confident it's internal)
df['y_score'] = signed_confidence(df['Judgment'], df['confidence'])

MEMORIES = ['SingleTurn', 'TrialChain']
print(f"Exp 1: {len(df):,} trials | memories: {df['memory'].unique().tolist()}")
print(df.groupby(['memory','source'])['accuracy'].agg(['count','mean']).round(3))


In [ ]:
# ── Compute SDT measures per model × memory ───────────────────────────────
# Type 2 AUC is restricted to perceived (external) trials only, per supplement.
# On imagined trials accuracy is 1.0 (models always say 'internal' for imagined
# and they are always correct), so including them produces degenerate AUC.
rows = []
for mem in MEMORIES:
    for model in MODEL_ORDER:
        sub = df[(df['memory'] == mem) & (df['model'] == model)]
        if sub.empty:
            continue
        sub_perc = sub[sub['source'] == 'perceived']   # perceived trials only for AUC2
        res = compute_sdt(sub['y_true'], sub['y_pred'], sub['y_score'],
                          accuracy=sub_perc['accuracy'], confidence=sub_perc['confidence'])
        rows.append(dict(memory=mem, model=model, **res))

sdt1 = pd.DataFrame(rows)

pd.set_option('display.float_format', '{:.3f}'.format)
print("\n=== Exp 1 SDT Summary ===")
print(sdt1[['memory','model','n_signal','n_noise',
            'HR','FA','dprime','c','auc1','auc2']].to_string(index=False))
print()
print("Note: auc1 = Type 1 AUC (source discrimination via signed confidence)")
print("      auc2 = Type 2 AUC (metacognitive on perceived trials only)")
print("      NaN  = all perceived trials correct → Type 2 AUC undefined")


In [ ]:
# ── Print APA-style SDT summary table ────────────────────────────────────
import warnings
dp_hdr = "d'"
print("\nTable S1. SDT Measures — Experiment 1")
print(f"{'Model':<20} {'Memory':<12} {'HR':>6} {'FA':>6} {dp_hdr:>7} {'c':>7} {'AUC1':>7} {'AUC2':>7}")
print('-' * 79)
for _, row in sdt1.iterrows():
    a1 = f"{row['auc1']:>7.3f}" if row['auc1'] is not None else f"{'N/A':>7}"
    import math
    a2 = f"{row['auc2']:>7.3f}" if (row['auc2'] is not None and not math.isnan(row['auc2'])) else f"{'NaN':>7}"
    print(f"{row['model']:<20} {row['memory']:<12} "
          f"{row['HR']:>6.3f} {row['FA']:>6.3f} "
          f"{row['dprime']:>7.3f} {row['c']:>7.3f} {a1} {a2}")
print()
print("Note. HR/FA = hit/false-alarm rate (log-linear corrected; Hautus, 1995).")
print("AUC1 = Type 1 AUC: source discrimination via signed confidence.")
print("AUC2 = Type 2 AUC: metacognitive sensitivity (confidence ~ correctness).")
print("NaN  = all trials correct in that cell (Type 2 AUC undefined).")


In [ ]:
# ── FigS7: SDT measures — Experiment 1 ──────────────────────────────────
# Layout: 1 row × 4 panels  (d', criterion c, Type 1 AUC, Type 2 AUC)
# Two grouped bars per model: SingleTurn (blue) / TrialChain (orange)
# NaN bars (Type 2 AUC undefined) are skipped — models with perfect accuracy.

import math

MEM_PAL = {'SingleTurn': '#4C72B0', 'TrialChain': '#DD8452'}
x   = np.arange(len(MODEL_ORDER))
bw  = 0.38

fig, axes = plt.subplots(1, 4, figsize=(18, 5))

panels = [
    ('dprime', "Sensitivity  $d'$",              None, None),
    ('c',      "Criterion  $c$",                 None, None),
    ('auc1',   "Type 1 AUC  (source discrim.)",  0.4,  1.05),
    ('auc2',   "Type 2 AUC  (metacognitive)",    0.2,  1.05),
]

for col, (metric, ylabel, ymin, ymax) in enumerate(panels):
    ax = axes[col]
    _panel_tag(ax, 'abcd'[col])
    for mi, mem in enumerate(MEMORIES):
        sub = sdt1[sdt1['memory'] == mem].set_index('model').reindex(MODEL_ORDER)
        vals = sub[metric].values.astype(float)
        offset = (mi - 0.5) * bw
        # Plot bars; use alpha=0.3 for NaN cells to signal undefined
        for xi, (v, mdl) in enumerate(zip(vals, MODEL_ORDER)):
            if math.isnan(v):
                ax.bar(xi + offset, 0, width=bw, color=MEM_PAL[mem],
                       alpha=0.15, edgecolor='grey', linewidth=0.5,
                       hatch='///')
            else:
                ax.bar(xi + offset, v, width=bw, color=MEM_PAL[mem],
                       alpha=0.85, edgecolor='white', linewidth=0.5,
                       label=mem if xi == 0 else '')

    ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
    if 'auc' in metric:
        ax.axhline(0.5, color='grey', linewidth=0.8, linestyle=':', alpha=0.7,
                   label='Chance (0.5)')

    ax.set_ylabel(ylabel)
    ax.set_xticks(x)
    ax.set_xticklabels(MODEL_LABELS, fontsize=9)
    ax.set_xlabel('Model')
    if ymin is not None:
        ax.set_ylim(ymin, ymax)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Shared "Memory" legend placed below all panels — avoids overlapping bars
# (previously drawn inside panel a, where it collided with tall d' bars).
mem_handles = [plt.Rectangle((0,0),1,1, color=MEM_PAL[m], alpha=0.85)
               for m in MEMORIES]
fig.legend(mem_handles, MEMORIES, title='Memory', loc='lower center',
           bbox_to_anchor=(0.5, -0.06), ncol=2, fontsize=9)

# Annotation: NaN = perfect accuracy
axes[3].text(0.5, 0.02, "Hatched = all trials correct (Type 2 AUC undefined)",
             transform=axes[3].transAxes, ha='center', fontsize=7.5,
             color='grey', style='italic')

plt.tight_layout()

for fmt in ('pdf','png'):
    fig.savefig(SUP / f'FigS7_exp1_sdt.{fmt}', dpi=DPI,
                bbox_inches='tight', facecolor='white')
print("Saved FigS7_exp1_sdt.pdf / .png")
plt.show()


In [ ]:
# ── FigS8: ROC Curves — Experiment 1 ────────────────────────────────────
# One subplot per memory type; 6 model curves each

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

for col, mem in enumerate(MEMORIES):
    ax = axes[col]
    _panel_tag(ax, 'ab'[col], title=mem.replace('SingleTurn','Single-Turn'))

    for mi, model in enumerate(MODEL_ORDER):
        sub = df[(df['memory'] == mem) & (df['model'] == model)]
        if sub.empty:
            continue
        fpr, tpr, _ = roc_curve(sub['y_true'], sub['y_score'])
        auc_val = roc_auc_score(sub['y_true'], sub['y_score'])
        ax.plot(fpr, tpr, color=MODEL_COLORS[mi], linewidth=1.8,
                label=f"{MODEL_LABELS[mi].replace(chr(10),' ')} (AUC={auc_val:.2f})")

    ax.plot([0,1],[0,1], 'k--', linewidth=0.8, alpha=0.5)
    ax.set_xlabel('False Alarm Rate  (1 − Specificity)')
    if col == 0:
        ax.set_ylabel('Hit Rate  (Sensitivity)')
    ax.legend(fontsize=8, loc='lower right')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()

for fmt in ('pdf','png'):
    fig.savefig(SUP / f'FigS8_exp1_roc.{fmt}', dpi=DPI,
                bbox_inches='tight', facecolor='white')
print("Saved FigS8_exp1_roc.pdf / .png")
plt.show()
